0. Load Libraries + Install API

In [1]:
#load libraries
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.font_manager as fm
import glob
import string
from tabulate import tabulate
import os
from pathlib import Path
import shutil
import geopandas as gpd
from shapely.geometry import Point, Polygon, LineString, box
from tqdm import tqdm
from matplotlib.lines import Line2D
import matplotlib.colors as mcolors
from matplotlib.colors import TwoSlopeNorm
import math
import time
from shapely.ops import unary_union
import seaborn as sns
from scipy.spatial import cKDTree
import matplotlib.cm as cm
from matplotlib.patches import Circle
from collections import defaultdict
import pyomo.environ as pyo
import gurobipy as gp
import osmnx as ox, networkx as nx
from osmnx.distance import nearest_nodes
from descartes import PolygonPatch
import requests
import matplotlib.colors as colors
import pickle
from datetime import datetime
import os
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

today = datetime.today().strftime("%Y%m%d")

1. Load Data

In [2]:
ROOT_DIR = Path.cwd()
DATA_DIR = ROOT_DIR / 'data'
RESULTS_DIR = ROOT_DIR / 'results'

In [3]:
# 격자단위 인구 (출처 : 통계청, 2023)
txt_files = list((DATA_DIR / 'grid_population_data_500').glob('*.txt'))
df_pop = pd.DataFrame(columns=['year', 'GRID_500M_', 'stat', 'population'])
for file_path in txt_files:
    df = pd.read_csv(file_path,sep="^", header = None)
    df.rename(columns={0:'year', 1:'GRID_500M_', 2:'stat', 3: 'population'}, inplace=True)
    df = df[df['stat'] == 'to_in_001']
    df_pop = pd.concat([df_pop, df], ignore_index = True)
df_pop

,year,GRID_500M_,stat,population
0,2023,가다78a67a,to_in_001,28
1,2023,가다78b68b,to_in_001,5
2,2023,가다79a67b,to_in_001,5
3,2023,가다79b68a,to_in_001,13
4,2023,가다79b68b,to_in_001,5
...,...,...,...,...
198083,2023,사사01b50b,to_in_001,38
198084,2023,사사02a48b,to_in_001,5
198085,2023,사사03b53a,to_in_001,0
198086,2023,사사87a24b,to_in_001,0


In [4]:
# 격자단위 경계 (출처 : 통계청, 2023)
GRID_BORDER_DIR = DATA_DIR / 'grid_border_500'
shp_files = list(GRID_BORDER_DIR.glob('*.shp'))
gdf_list = []
for shp_file in shp_files:
    gdf = gpd.read_file(shp_file)
    gdf_list.append(gdf)
df_grid = gpd.GeoDataFrame(pd.concat(gdf_list, ignore_index=True))
df_grid = df_grid.drop_duplicates()
df_grid

,GRID_500M_,geometry
0,가다77b65b,"POLYGON ((777500 1565500, 777500 1566000, 7780..."
1,가다77b66b,"POLYGON ((777500 1566500, 777500 1567000, 7780..."
2,가다77b67a,"POLYGON ((777500 1567000, 777500 1567500, 7780..."
3,가다77b67b,"POLYGON ((777500 1567500, 777500 1568000, 7780..."
4,가다77b69b,"POLYGON ((777500 1569500, 777500 1570000, 7780..."
...,...,...
513005,사사01a53b,"POLYGON ((1301000 1953500, 1301000 1954000, 13..."
513006,사사01a54a,"POLYGON ((1301000 1954000, 1301000 1954500, 13..."
513007,사사01a54b,"POLYGON ((1301000 1954500, 1301000 1955000, 13..."
513008,사사01a55a,"POLYGON ((1301000 1955000, 1301000 1955500, 13..."


In [5]:
# 읍면동 정보 (출처 : GEO-SERVICE WEB 아카이브, 2025/03)
file_path = DATA_DIR / 'emd_GIS_data/emd.shp'
gdf_emd = gpd.read_file(file_path, encoding="euc-kr")

# Synchronize coordinate reference system
df_grid = df_grid.to_crs(gdf_emd.crs)

df_grid

,GRID_500M_,geometry
0,가다77b65b,"POLYGON ((777500 1565500, 777500 1566000, 7780..."
1,가다77b66b,"POLYGON ((777500 1566500, 777500 1567000, 7780..."
2,가다77b67a,"POLYGON ((777500 1567000, 777500 1567500, 7780..."
3,가다77b67b,"POLYGON ((777500 1567500, 777500 1568000, 7780..."
4,가다77b69b,"POLYGON ((777500 1569500, 777500 1570000, 7780..."
...,...,...
513005,사사01a53b,"POLYGON ((1301000 1953500, 1301000 1954000, 13..."
513006,사사01a54a,"POLYGON ((1301000 1954000, 1301000 1954500, 13..."
513007,사사01a54b,"POLYGON ((1301000 1954500, 1301000 1955000, 13..."
513008,사사01a55a,"POLYGON ((1301000 1955000, 1301000 1955500, 13..."


In [6]:
# 의료기관 정보 (출처 : 건강보험심사평가원, 2025/03)
df_inst = pd.read_excel(DATA_DIR / '2025_03_institute_info.xlsx')
df_inst

,암호화요양기호,요양기관명,종별코드,종별코드명,시도코드,시도코드명,시군구코드,시군구코드명,읍면동,우편번호,...,치과인턴 인원수,치과레지던트 인원수,치과전문의 인원수,한방일반의 인원수,한방인턴 인원수,한방레지던트 인원수,한방전문의 인원수,조산사 인원수,좌표(X),좌표(Y)
0,JDQ4MTYyMiM4MSMkMiMkMCMkMDAkNDgxOTYxIzIxIyQxIy...,(VOM)봄안과의원,31,의원,230000,대구,230002,대구동구,율하동,41151,...,0,0,0,0,0,0,0,0,128.691894,35.869852
1,JDQ4MTAxMiM1MSMkMSMkMCMkOTkkNTgxOTYxIzExIyQxIy...,(복지)예사랑병원,29,정신병원,210000,부산,210006,부산서구,아미동2가,49249,...,0,0,0,0,0,0,0,0,129.013310,35.097325
2,JDQ4MTYyMiM4MSMkMiMkMCMkMDAkNDgxOTYxIzUxIyQxIy...,(사) 경찰공제회 포항의원,31,의원,370000,경북,370701,포항남구,오천읍,37895,...,0,0,0,0,0,0,0,0,129.392904,35.942317
3,JDU4MTI3MSM1MSMkMiMkMCMkMDAkNDgxOTYxIzIxIyQxIy...,(사)경찰공제회남부의원,31,의원,220000,인천,220006,인천남동구,고잔동,21689,...,0,0,0,0,0,0,0,0,126.708861,37.384910
4,JDQ4MTYyMiM1MSMkMiMkMCMkMDAkMzgxMzUxIzQxIyQxIy...,(사)대한결핵협회경기도지부부설복십자의원,31,의원,310000,경기,310603,수원팔달구,매산로3가,16456,...,0,0,0,0,0,0,0,0,127.008966,37.271984
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
78769,JDQ4MTg4MSM1MSMkMiMkNCMkMDAkNDgxMTkxIzMxIyQxIy...,힘튼의원,31,의원,110000,서울,110015,은평구,응암동,3462,...,0,0,0,0,0,0,0,0,126.926666,37.601101
78770,JDQ4MTYyMiM2MSMkMSMkMiMkNzIkMzgxMTkxIzExIyQxIy...,힘플란트치과의원,51,치과의원,340000,충남,340500,아산시,둔포면,31406,...,0,0,0,0,0,0,0,0,127.057823,36.923728
78771,JDQ4MTYyMiM2MSMkMSMkMiMkNzIkMzgxOTYxIzExIyQxIy...,힘플란트치과의원,51,치과의원,250000,대전,250003,대전서구,내동,35314,...,0,0,0,0,0,0,0,0,127.383241,36.332062
78772,JDQ4MTYyMiM3MSMkMSMkMCMkOTkkMzgxMzUxIzIxIyQxIy...,힘플러스병원,21,병원,360000,전남,360400,순천시,조례동,57946,...,0,0,0,0,0,0,0,0,127.519845,34.960448


In [7]:
# 읍면동별 인구 데이터 (출처 : 통계청, 2025/03)
df_emd_pop = pd.read_csv(DATA_DIR / '지역별(법정동) 성별 연령별 주민등록 인구수_20250331.csv', encoding='euc-kr', low_memory=False)
# encoding : 문자 한글

# 법정동 코드 : 10자리 숫자, 1-2 : 시도, 3-5 : 시군구, 6-8 : 읍면동, 9-10 : 리
# 법정동코드는 8자리에서 끊어야 읍면동 단위임
df_emd_pop['법정동코드'] = df_emd_pop['법정동코드'].astype(str).str[:8]

# 리로 분리되는 동 확인
#df_emd_pop['법정동코드'] = df_emd_pop['법정동코드'].astype(str).str[8:10]
#df_emd_pop = df_emd_pop[df_emd_pop['법정동코드'] != '00']

df_emd_pop

,법정동코드,기준연월,시도명,시군구명,읍면동명,리명,계,남자,여자,0세남자,...,101세여자,102세여자,103세여자,104세여자,105세여자,106세여자,107세여자,108세여자,109세여자,110세이상 여자
0,11110101,2025-03-31,서울특별시,종로구,청운동,NaN,2227,1094,1133,6,...,0,0,0,1,0,0,0,0,0,0
1,11110102,2025-03-31,서울특별시,종로구,신교동,NaN,1277,566,711,3,...,0,0,0,0,0,0,0,0,0,0
2,11110103,2025-03-31,서울특별시,종로구,궁정동,NaN,218,94,124,0,...,0,0,0,0,0,0,0,0,0,0
3,11110104,2025-03-31,서울특별시,종로구,효자동,NaN,550,252,298,0,...,0,0,0,0,0,0,0,0,0,0
4,11110105,2025-03-31,서울특별시,종로구,창성동,NaN,379,167,212,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18705,52800420,2025-03-31,전북특별자치도,부안군,위도면,대리,205,99,106,0,...,0,0,0,0,0,0,0,0,0,0
18706,52800420,2025-03-31,전북특별자치도,부안군,위도면,거륜리,9,7,2,0,...,0,0,0,0,0,0,0,0,0,0
18707,52800420,2025-03-31,전북특별자치도,부안군,위도면,식도리,140,83,57,0,...,0,0,0,0,0,0,0,0,0,0
18708,52800420,2025-03-31,전북특별자치도,부안군,위도면,상왕등리,47,29,18,0,...,0,0,0,0,0,0,0,0,0,0


In [8]:
# 시군구, 나이대, 성별 별 건강상태 집단 비율 (출처 : 질병관리청 지역사회 건강조사 2024로부터 가공된 파일)
df_rate_gender_age_sgg = pd.read_csv(DATA_DIR / 'rate_gender_age_signgu.csv', low_memory=False)
df_rate_gender_age_sgg

,signgu_code,AGE_GROUP,SEX,시군구보건소,고O 당O,고O 당X,고X 당O,고X 당X
0,11110,3,1,11110 서울 종로구보건소 20-29 남,0.000000,0.091466,0.000000,0.908534
1,11110,3,2,11110 서울 종로구보건소 20-29 여,0.000000,0.000000,0.018995,0.981005
2,11110,4,1,11110 서울 종로구보건소 30-39 남,0.000000,0.049477,0.014844,0.935679
3,11110,4,2,11110 서울 종로구보건소 30-39 여,0.000000,0.017365,0.000000,0.982635
4,11110,5,1,11110 서울 종로구보건소 40-49 남,0.026166,0.117417,0.021131,0.835286
...,...,...,...,...,...,...,...,...
3951,52800,8,1,52800 전북 부안군보건소 70-79 남,0.177231,0.418882,0.048525,0.355362
3952,52800,8,2,52800 전북 부안군보건소 70-79 여,0.242489,0.450991,0.054739,0.251781
3953,52800,9,1,52800 전북 부안군보건소 80+ 남,0.232223,0.393344,0.041482,0.332951
3954,52800,9,2,52800 전북 부안군보건소 80+ 여,0.252292,0.529273,0.037740,0.180694


In [9]:
# 비연륙도서 소속 기관들
df_bogun_island = pd.read_excel(DATA_DIR / '비연륙도서 라벨링.xlsx', sheet_name="비연륙 (24배치)")
df_bogun_island

,라벨,암호화요양기호,요양기관명,종별코드,종별코드명,시도코드,시도코드명,시군구코드,시군구코드명,읍면동,...,치과인턴 인원수,치과레지던트 인원수,치과전문의 인원수,한방일반의 인원수,한방인턴 인원수,한방레지던트 인원수,한방전문의 인원수,조산사 인원수,좌표(X),좌표(Y)
0,비연륙 (24배치),JDQ4MTYyMiM3MSMkMSMkMCMkOTIkMzgxOTYxIzIxIyQyIy...,가거도보건지소,72,보건지소,360000,전남,360012,신안군,흑산면,...,0,0,0,1,0,0,0,0,125.126635,34.052378
1,비연륙 (24배치),JDQ4MTYyMiM3MSMkMSMkMCMkOTIkMzgxOTYxIzExIyQyIy...,개도보건지소,72,보건지소,360000,전남,360500,여수시,화정면,...,0,0,0,1,0,0,0,0,127.663759,34.571450
2,비연륙 (24배치),JDQ4MTYyMiM2MSMkMSMkNiMkOTIkMzgxOTYxIzExIyQxIy...,군산시개야도보건지소,72,보건지소,350000,전북,350100,군산시,옥도면,...,0,0,0,0,0,0,0,0,126.552899,36.037516
3,비연륙 (24배치),JDQ4MTYyMiM2MSMkMSMkNiMkOTIkMzgxOTYxIzExIyQxIy...,군산시어청도보건지소,72,보건지소,350000,전북,350100,군산시,옥도면,...,0,0,0,0,0,0,0,0,125.979973,36.120939
4,비연륙 (24배치),JDQ4MTYyMiM3MSMkMSMkMCMkOTIkMzgxOTYxIzExIyQxIy...,낙월보건지소,72,보건지소,360000,전남,360014,영광군,낙월면,...,0,0,0,1,0,0,0,0,126.140068,35.202467
5,비연륙 (24배치),JDQ4MTYyMiM3MSMkMSMkMCMkOTIkMzgxOTYxIzExIyQyIy...,남면보건지소,72,보건지소,360000,전남,360500,여수시,남면,...,0,0,0,1,0,0,0,0,127.767154,34.509220
6,비연륙 (24배치),JDQ4MTYyMiM1MSMkMSMkMCMkOTIkMzgxOTYxIzExIyQxIy...,대청보건지소,72,보건지소,220000,인천,220200,인천옹진군,대청면,...,0,0,0,0,0,0,1,0,124.715493,37.824416
7,비연륙 (24배치),JDQ4MTYyMiM1MSMkMSMkMCMkOTIkMzgxOTYxIzExIyQxIy...,덕적보건지소,72,보건지소,220000,인천,220200,인천옹진군,덕적면,...,0,0,0,1,0,0,0,0,126.147107,37.226183
8,비연륙 (24배치),JDQ4MTYyMiM3MSMkMSMkMCMkOTIkMzgxOTYxIzExIyQxIy...,도초면보건지소,72,보건지소,360000,전남,360012,신안군,도초면,...,0,0,0,1,0,0,0,0,125.954611,34.703912
9,비연륙 (24배치),JDQ4MTYyMiM1MSMkMSMkMCMkOTIkMzgxOTYxIzExIyQxIy...,백령보건지소,72,보건지소,220000,인천,220200,인천옹진군,백령면,...,0,0,0,1,0,0,0,0,124.664719,37.958025


In [10]:
# 오픈 소스 지도 데이터 (출처: OpenStreetMap, Live Update)
data_dir = DATA_DIR / "openstreetmap_region"
with open(DATA_DIR/"openstreetmap_region/G_한국.pkl", "rb") as f:
    G_한국 = pickle.load(f)
with open(DATA_DIR/"openstreetmap_region/G_울릉.pkl", "rb") as f:
    G_울릉 = pickle.load(f)
with open(DATA_DIR/"openstreetmap_region/G_제주.pkl", "rb") as f:
    G_제주 = pickle.load(f)

In [11]:
# 코호트 2.0을 활용한 코호트별 최소의료 연간 이용률
df_cohort_final_summary = pd.read_csv(DATA_DIR / 'cohort_final_summary.csv', low_memory=False)
df_cohort_final_summary

,NEW_AGE_GROUP,COND,SEX,MEAN_COUNT,MEDIAN_COUNT,NUM
0,1,0,1,7.957718,5.0,2436.0
1,1,0,2,7.467028,5.0,2214.0
2,1,1,1,NaN,NaN,NaN
3,1,1,2,NaN,NaN,NaN
4,1,2,1,NaN,NaN,NaN
...,...,...,...,...,...,...
75,9,1,2,23.786851,18.0,16777.0
76,9,2,1,18.185680,13.0,3366.0
77,9,2,2,22.208251,16.0,3563.0
78,9,3,1,22.867052,17.0,3806.0


In [12]:
# 공보의 현황 (출처: 보건복지부, 2025/08)
df_current_allocation = pd.read_excel(DATA_DIR / '251030_public_health_doctor_stats.xlsx')
df_current_allocation['비연륙도서 여부'] = df_current_allocation['비연륙도서 여부'].fillna(0)
df_current_allocation

,시도코드명,시군구코드명,읍면동,요양기관명,비연륙도서 여부,현재공보의배치여부,2026년 전역,2027년 전역,2028년 전역
0,강원,강릉시,사천면,사천보건지소,0.0,1,0,1,0
1,강원,강릉시,옥계면,옥계보건지소,0.0,1,0,1,0
2,강원,강릉시,주문진읍,주문진통합보건지소,0.0,1,0,0,1
3,강원,고성군,죽왕면,죽왕보건지소,0.0,1,1,0,0
4,강원,동해시,이로동,삼화보건지소,0.0,1,1,0,0
...,...,...,...,...,...,...,...,...,...
604,충북,영동군,NaN,영동군보건소,0.0,1,0,0,1
605,충북,음성군,NaN,음성군보건소,0.0,1,0,0,1
606,충북,증평군,NaN,증평군보건소,0.0,1,0,0,1
607,충북,충주시,NaN,충주시보건소,0.0,1,1,0,0


2. 최소의료 수요 산출

1) 최소의료 범주를 바탕으로 한 성/연령/건강상태 corort 별 최소의료 연간 이용률 추정정

In [13]:
# data preprocessing
complete_df = df_cohort_final_summary.fillna(0)
complete_df['NUM'] = complete_df['NUM'].astype(int)
complete_df['NEW_AGE_GROUP'] = complete_df['NEW_AGE_GROUP'].astype(str)
complete_df['COND'] = complete_df['COND'].astype(str)
complete_df['SEX'] = complete_df['SEX'].astype(str)

complete_df = complete_df.rename(columns = {
    'NEW_AGE_GROUP' : 'AGE_GROUP',
    'COND' : 'CONDITION',
    'MEAN_COUNT' : 'avg_visit',
    'NUM' : 'population_count'
})

complete_df

,AGE_GROUP,CONDITION,SEX,avg_visit,MEDIAN_COUNT,population_count
0,1,0,1,7.957718,5.0,2436
1,1,0,2,7.467028,5.0,2214
2,1,1,1,0.000000,0.0,0
3,1,1,2,0.000000,0.0,0
4,1,2,1,0.000000,0.0,0
...,...,...,...,...,...,...
75,9,1,2,23.786851,18.0,16777
76,9,2,1,18.185680,13.0,3366
77,9,2,2,22.208251,16.0,3563
78,9,3,1,22.867052,17.0,3806


In [14]:
# group cohort as same age group and condition
grouped = (
    complete_df[['SEX', 'AGE_GROUP', 'CONDITION', 'population_count']]
    .groupby(['AGE_GROUP', 'CONDITION'])['population_count']
    .sum()
    .reset_index()
)

grouped['percentage'] = (
    grouped['population_count'] / grouped.groupby('AGE_GROUP')['population_count'].transform('sum')
) * 100

grouped = grouped.sort_values(by='AGE_GROUP', ascending=True)
grouped

,AGE_GROUP,CONDITION,population_count,percentage
0,1,0,4650,100.000000
1,1,1,0,0.000000
2,1,2,0,0.000000
3,1,3,0,0.000000
4,10,0,9663,28.178584
5,10,1,17221,50.218710
6,10,2,2681,7.818150
7,10,3,4727,13.784556
11,2,3,0,0.000000
10,2,2,9,0.011740


2) 각 읍면동 인구 데이터에 대입

In [15]:
# 읍면동 인구 데이터는 1세 단위로 있음 -> Age Group 기준에 맞게 aggregate
def aggregate_age_groups(df, sex_label, prefix):
    age_group_map = {
        '1': [f'{i}세{prefix}' for i in range(0, 1)], # -> 0세는 Age Group 1 이다
        '2': [f'{i}세{prefix}' for i in range(1, 10)], # -> 1-9세는 Age Group 2 이다
        '3': [f'{i}세{prefix}' for i in range(10, 19)],
        '19': [f'{i}세{prefix}' for i in range(19, 20)], # 19세는 당뇨, 고혈압 유병률 정보가 있기 때문에 우선 따로 계산 후, 나중에 '3'에 포함시킴
        '4': [f'{i}세{prefix}' for i in range(20, 30)],
        '5': [f'{i}세{prefix}' for i in range(30, 40)],
        '6': [f'{i}세{prefix}' for i in range(40, 50)],
        '7': [f'{i}세{prefix}' for i in range(50, 60)],
        '8': [f'{i}세{prefix}' for i in range(60, 70)],
        '9': [f'{i}세{prefix}' for i in range(70, 80)],
        '10': [f'{i}세{prefix}' for i in range(80, 111)] + [f'110세이상 {prefix}']
    }

    rows = []
    for group_id, cols in age_group_map.items():
        valid_cols = [col for col in cols if col in df.columns]
        group_df = pd.DataFrame({
            '법정동코드': df['법정동코드'],
            'SEX': sex_label,
            'AGE_GROUP': group_id,
            'population': df[valid_cols].sum(axis=1)
        })
        rows.append(group_df)
    return pd.concat(rows, axis=0, ignore_index=True)

In [16]:
# 성별, 나이대별 aggregation
male_df = aggregate_age_groups(df_emd_pop, sex_label=1, prefix='남자')
female_df = aggregate_age_groups(df_emd_pop, sex_label=2, prefix='여자')
df_emd_pop_agegroup = pd.concat([male_df, female_df], ignore_index=True)
df_emd_pop_agegroup['SEX'] = df_emd_pop_agegroup['SEX'].astype(str)
df_emd_pop_agegroup = df_emd_pop_agegroup.sort_values(by = ['법정동코드', 'AGE_GROUP']).reset_index(drop = True)
df_emd_pop_agegroup

,법정동코드,SEX,AGE_GROUP,population
0,11110101,1,1,6
1,11110101,2,1,4
2,11110101,1,10,61
3,11110101,2,10,79
4,11110101,1,19,17
...,...,...,...,...
411615,52800420,2,9,25
411616,52800420,2,9,0
411617,52800420,2,9,16
411618,52800420,2,9,3


In [17]:
# df_emd_needweights에 정보를 담음
df_emd_needweights = df_emd_pop_agegroup.copy()
df_emd_needweights['SGG_CD'] = df_emd_needweights['법정동코드'].astype(str).str[:5]

# df_rate_gender_age_sgg 프로세싱 -> 건강상태 구분을 위한 정보
# df_rate_gender_age_sgg에서 AGE_GROUP = 2 는 19세의 정보임
df_rate_gender_age_sgg['SGG_CD'] = df_rate_gender_age_sgg['signgu_code'].astype(str)
df_rate_gender_age_sgg['SEX'] = df_rate_gender_age_sgg['SEX'].astype(str)
df_rate_gender_age_sgg['AGE_GROUP'] = df_rate_gender_age_sgg['AGE_GROUP'].astype(str)
df_rate_gender_age_sgg.loc[df_rate_gender_age_sgg['AGE_GROUP'] == '2', 'AGE_GROUP'] = '19'
df_emd_needweights = pd.merge(df_emd_needweights, df_rate_gender_age_sgg[['SGG_CD', 'SEX', 'AGE_GROUP', '고O 당O', '고O 당X', '고X 당O', '고X 당X']], on = ['SGG_CD', 'SEX', 'AGE_GROUP'], how = 'left')

# 미성년자 (AGE_GROUP = 1 or 2)는 전부 고X당X로 바꾸자
df_emd_needweights = df_emd_needweights.fillna({'고O 당O':0, '고O 당X':0, '고X 당O':0, '고X 당X':1})

# 각 카테고리 별로 예상 방문횟수 구하기
categories = ['고O 당O', '고O 당X', '고X 당O', '고X 당X']
for column in categories:
    df_emd_needweights[column] = df_emd_needweights[column]*df_emd_needweights['population']

# AGE_GROUP = 19와 AGE_GROUP = 3 다시 합치기
df_emd_needweights.loc[df_emd_needweights['AGE_GROUP']=='19', 'AGE_GROUP'] = '3'
df_emd_needweights = df_emd_needweights.groupby(['법정동코드', 'SEX', 'AGE_GROUP'], as_index = False)[['population', '고O 당O', '고O 당X', '고X 당O', '고X 당X']].sum()

# Multiply avg_visits
condition_map = {
    '고X 당X': '0',
    '고O 당X': '1',
    '고X 당O': '2',
    '고O 당O': '3'
}
for col_name, condition_id in condition_map.items():
    df_merge = complete_df[complete_df['CONDITION'] == condition_id][['AGE_GROUP', 'SEX', 'avg_visit']]
    df_emd_needweights = df_emd_needweights.merge(df_merge, on = ['AGE_GROUP', 'SEX'], how = 'left')
    df_emd_needweights[col_name] = df_emd_needweights[col_name]*df_emd_needweights['avg_visit']
    df_emd_needweights = df_emd_needweights.drop(columns = ['avg_visit'])

# 연령대별로 분할되어 있는 df 하나 저장해놓기
df_emd_needweights_agegroupsplit = df_emd_needweights.sort_values('AGE_GROUP').copy()
df_emd_needweights_agegroupsplit

,법정동코드,SEX,AGE_GROUP,population,고O 당O,고O 당X,고X 당O,고X 당X
0,11110101,1,1,6,0.000000,0.000000,0.000000,47.746305
58310,46150320,2,1,53,0.000000,0.000000,0.000000,395.752484
58300,46150320,1,1,52,0.000000,0.000000,0.000000,413.801314
58290,46150310,2,1,197,0.000000,0.000000,0.000000,1471.004517
13900,27110150,1,1,0,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...
60739,46780253,2,9,1061,6180.075944,11342.023823,563.235398,5585.476652
60729,46780253,1,9,904,4170.812459,8371.112838,813.938110,3073.030425
60719,46780250,2,9,558,3250.219017,5964.985196,296.216166,2937.507985
60779,46780320,2,9,118,687.322301,1261.412640,62.640695,621.193445


3) 보정계수 및 보정인구수 산출

In [18]:
# Aggregate 읍면동
df_emd_needweights = df_emd_needweights.groupby(['법정동코드'], as_index = False)[['population', '고O 당O', '고O 당X', '고X 당O', '고X 당X']].sum()
df_emd_needweights['expected_visits'] = df_emd_needweights[categories].sum(axis = 1)
df_emd_needweights

,법정동코드,population,고O 당O,고O 당X,고X 당O,고X 당X,expected_visits
0,11110101,2227,2727.977812,5091.815305,988.468886,12875.830125,21684.092127
1,11110102,1277,1494.303264,2827.995723,600.078275,7084.670309,12007.047572
2,11110103,218,180.591497,357.568360,92.177900,1220.881993,1851.219750
3,11110104,550,721.934167,1279.037463,257.307000,3119.457476,5377.736106
4,11110105,379,579.576156,1082.304116,186.512502,2144.832208,3993.224982
...,...,...,...,...,...,...,...
5008,52800380,2313,6323.734494,10575.002118,1275.927883,13378.324670,31552.989165
5009,52800390,2021,5580.760484,9341.304729,1154.718026,10954.028290,27030.811529
5010,52800400,2353,6579.532945,10998.423877,1332.642567,13162.818068,32073.417456
5011,52800410,2407,6881.236410,11455.806661,1403.940351,12747.304870,32488.288292


In [19]:
# Expected visits per person
df_emd_needweights['expected_visits_per_person'] = df_emd_needweights['expected_visits'] / df_emd_needweights['population']
df_emd_needweights[['법정동코드','expected_visits_per_person']]

,법정동코드,expected_visits_per_person
0,11110101,9.736907
1,11110102,9.402543
2,11110103,8.491834
3,11110104,9.777702
4,11110105,10.536214
...,...,...
5008,52800380,13.641586
5009,52800390,13.374969
5010,52800400,13.630862
5011,52800410,13.497419


In [20]:
# 전국 Expected Visits
korea_expected_visits = df_emd_needweights['expected_visits'].sum()
korea_expected_visits_per_person = korea_expected_visits/df_emd_needweights['population'].sum()

# 보정계수
df_emd_needweights['coefficient'] = df_emd_needweights['expected_visits_per_person']/korea_expected_visits_per_person

# 보정인구수
df_emd_needweights['adjusted_population'] = df_emd_needweights['population']*df_emd_needweights['coefficient']

df_emd_needweights[['법정동코드', 'adjusted_population', 'coefficient']]

,법정동코드,adjusted_population,coefficient
0,11110101,2136.972165,0.959574
1,11110102,1183.297244,0.926623
2,11110103,182.438124,0.836872
3,11110104,529.977105,0.963595
4,11110105,393.533222,1.038346
...,...,...,...
5008,52800380,3109.554192,1.344381
5009,52800390,2663.892567,1.318106
5010,52800400,3160.842518,1.343324
5011,52800410,3201.728132,1.330174


In [21]:
# Ensure both key columns are string type
gdf_emd['EMD_CD'] = gdf_emd['EMD_CD'].astype(str)
df_emd_needweights['법정동코드'] = df_emd_needweights['법정동코드'].astype(str)

# Merge on gdf
df_emd_needweights['EMD_CD'] = df_emd_needweights['법정동코드'].astype(str).str[:8]
gdf_emd_expected_visits_needweights = gpd.GeoDataFrame(pd.merge(df_emd_needweights, gdf_emd, on = 'EMD_CD', how='left'), geometry='geometry')
gdf_emd_expected_visits_needweights

,법정동코드,population,고O 당O,고O 당X,고X 당O,고X 당X,expected_visits,expected_visits_per_person,coefficient,adjusted_population,EMD_CD,EMD_ENG_NM,EMD_KOR_NM,geometry
0,11110101,2227,2727.977812,5091.815305,988.468886,12875.830125,21684.092127,9.736907,0.959574,2136.972165,11110101,Cheongun-dong,청운동,"POLYGON ((953700.022 1954605.065, 953693.871 1..."
1,11110102,1277,1494.303264,2827.995723,600.078275,7084.670309,12007.047572,9.402543,0.926623,1183.297244,11110102,Singyo-dong,신교동,"POLYGON ((953233.465 1953996.984, 953235.183 1..."
2,11110103,218,180.591497,357.568360,92.177900,1220.881993,1851.219750,8.491834,0.836872,182.438124,11110103,Gungjeong-dong,궁정동,"POLYGON ((953560.228 1954257.466, 953561.19 19..."
3,11110104,550,721.934167,1279.037463,257.307000,3119.457476,5377.736106,9.777702,0.963595,529.977105,11110104,Hyoja-dong,효자동,"POLYGON ((953519.843 1953890.785, 953518.489 1..."
4,11110105,379,579.576156,1082.304116,186.512502,2144.832208,3993.224982,10.536214,1.038346,393.533222,11110105,Changseong-dong,창성동,"POLYGON ((953516.123 1953734.362, 953516.526 1..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5008,52800380,2313,6323.734494,10575.002118,1275.927883,13378.324670,31552.989165,13.641586,1.344381,3109.554192,52800380,Baeksan-myeon,백산면,"POLYGON ((934492.435 1747398.266, 934498.181 1..."
5009,52800390,2021,5580.760484,9341.304729,1154.718026,10954.028290,27030.811529,13.374969,1.318106,2663.892567,52800390,Sangseo-myeon,상서면,"POLYGON ((924979.638 1740610.214, 924985.428 1..."
5010,52800400,2353,6579.532945,10998.423877,1332.642567,13162.818068,32073.417456,13.630862,1.343324,3160.842518,52800400,Haseo-myeon,하서면,"POLYGON ((924910.563 1749481.985, 924971.3 174..."
5011,52800410,2407,6881.236410,11455.806661,1403.940351,12747.304870,32488.288292,13.497419,1.330174,3201.728132,52800410,Julpo-myeon,줄포면,"POLYGON ((929554.188 1736493.399, 929554.106 1..."


4) 500m 격자 단위 보정

In [22]:
gdf_pop = df_pop.copy()
gdf_emd = gdf_emd.to_crs(df_grid.crs)

# centroid 불러오기
df_grid["grid_centroid"] = df_grid.geometry.centroid

# 인구정보 불러오기
gdf_grid_pop = df_grid.merge(
    gdf_pop[["GRID_500M_", "population"]], on="GRID_500M_", how="left"
)
gdf_grid_pop["population"] = gdf_grid_pop["population"].fillna(0).astype(int)

gdf_grid_pop[gdf_grid_pop['population']!=0][['GRID_500M_', 'population']]

C:\Users\hshoo\AppData\Local\Temp\ipykernel_20216\856185526.py:11: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  gdf_grid_pop["population"] = gdf_grid_pop["population"].fillna(0).astype(int)


,GRID_500M_,population
14,가다81a64b,256
25,가다79b68a,13
26,가다79b68b,5
33,가다78b68b,5
39,가다79a67b,5
...,...,...
418681,사사00a54a,5
418687,사사00a53a,5
418688,사사01a53b,5
418689,사사01a54a,5


In [23]:
# 우선, 격자 centroid를 기준으로 읍면동에 할당
pts = gdf_grid_pop.set_geometry("grid_centroid")[["GRID_500M_", "grid_centroid"]]
emd_attr = gpd.sjoin( # 공간 조인
    pts,
    gdf_emd[["EMD_CD", "geometry"]],
    how="left",
    predicate="covers"
)[["EMD_CD"]]
gdf_join1 = gdf_grid_pop.join(emd_attr)

# Centroid가 어떤 읍면동에도 포함되지 않은 경우, 근접 읍면동에 할당
unassigned_mask = gdf_join1["EMD_CD"].isna()
if unassigned_mask.any():
    left_idx = gdf_join1.index[unassigned_mask]

    nearest = gpd.sjoin_nearest(
        gdf_join1.loc[left_idx, ["GRID_500M_", "grid_centroid"]].set_geometry("grid_centroid"),
        gdf_emd[["EMD_CD", "geometry"]],
        how="left",
        max_distance=50,
        distance_col="dist"
    )

    # keep a single match per left row (in case of ties)
    nearest = (
        nearest[~nearest.index.duplicated(keep="first")]
        [["EMD_CD"]]
        .rename(columns={"EMD_CD": "EMD_CD_nearest"}) # temporary column
    )

    # index-aligned join, then fill NA from the nearest result
    gdf_join1 = gdf_join1.join(nearest, how="left")
    gdf_join1["EMD_CD"] = gdf_join1["EMD_CD"].fillna(gdf_join1["EMD_CD_nearest"])
    gdf_join1 = gdf_join1.drop(columns=["EMD_CD_nearest"])

# 여전히 할당 안된 격자는 면적 기반으로 가장 많이 겹치는 읍면동 할당
remaining_mask = gdf_join1["EMD_CD"].isna()
if remaining_mask.any():
    inter = gpd.overlay(
        gdf_grid_pop.loc[remaining_mask, ["GRID_500M_", "geometry"]],
        gdf_emd[["EMD_CD", "geometry"]],
        how="intersection"
    )
    inter["area"] = inter.geometry.area

    best = (
        inter.sort_values(["GRID_500M_", "area"], ascending=[True, False])
             .drop_duplicates("GRID_500M_")[["GRID_500M_", "EMD_CD"]]
             .rename(columns={"EMD_CD": "EMD_CD_area"})
    )

    # 왼쪽의 EMD_CD는 유지, 오른쪽 결과를 붙여서 NA만 채움
    gdf_join1 = gdf_join1.merge(best, on="GRID_500M_", how="left")
    gdf_join1["EMD_CD"] = gdf_join1["EMD_CD"].fillna(gdf_join1["EMD_CD_area"])
    gdf_join1 = gdf_join1.drop(columns=["EMD_CD_area"])

# Coefficient 및 adjusted population 계산
gdf_grid_pop = gdf_join1.merge(
    gdf_emd_expected_visits_needweights[["법정동코드", "coefficient"]],
    left_on="EMD_CD",
    right_on="법정동코드",
    how="left"
)
gdf_grid_pop["adjusted_population"] = (
    gdf_grid_pop["population"] * gdf_grid_pop["coefficient"]
)


In [24]:
# 확인 결과, 할당되지 않은 모든 격자에는 인구가 0명임 ..?
gdf_grid_pop = gdf_grid_pop.dropna().reset_index(drop=True)
# gdf_grid_pop[gdf_grid_pop['EMD_CD'].isna() & gdf_grid_pop['population'] > 0]

3. 도로망 거리 계산

In [25]:
# 기관 그룹 분리
grp_hosp   = {'의원', '병원', '종합병원', '보건의료원'} #한의원, 치과의원 제외외
grp_bogun  = {'보건소', '보건지소'}
target_crs = gdf_emd.crs # 기준 좌표계계
gdf_inst = gpd.GeoDataFrame(
    df_inst,
    geometry=gpd.points_from_xy(df_inst["좌표(X)"], df_inst["좌표(Y)"]),
    crs="EPSG:4326" # EPSG:4326 -> 경위도 좌표계"
)
gdf_inst = gdf_inst.to_crs(target_crs)
gdf_inst

,암호화요양기호,요양기관명,종별코드,종별코드명,시도코드,시도코드명,시군구코드,시군구코드명,읍면동,우편번호,...,치과레지던트 인원수,치과전문의 인원수,한방일반의 인원수,한방인턴 인원수,한방레지던트 인원수,한방전문의 인원수,조산사 인원수,좌표(X),좌표(Y),geometry
0,JDQ4MTYyMiM4MSMkMiMkMCMkMDAkNDgxOTYxIzIxIyQxIy...,(VOM)봄안과의원,31,의원,230000,대구,230002,대구동구,율하동,41151,...,0,0,0,0,0,0,0,128.691894,35.869852,POINT (1107601.3 1764354.058)
1,JDQ4MTAxMiM1MSMkMSMkMCMkOTkkNTgxOTYxIzExIyQxIy...,(복지)예사랑병원,29,정신병원,210000,부산,210006,부산서구,아미동2가,49249,...,0,0,0,0,0,0,0,129.013310,35.097325,POINT (1137933.843 1679068.558)
2,JDQ4MTYyMiM4MSMkMiMkMCMkMDAkNDgxOTYxIzUxIyQxIy...,(사) 경찰공제회 포항의원,31,의원,370000,경북,370701,포항남구,오천읍,37895,...,0,0,0,0,0,0,0,129.392904,35.942317,POINT (1170736.806 1773391.401)
3,JDU4MTI3MSM1MSMkMiMkMCMkMDAkNDgxOTYxIzIxIyQxIy...,(사)경찰공제회남부의원,31,의원,220000,인천,220006,인천남동구,고잔동,21689,...,0,0,0,0,0,0,0,126.708861,37.384910,POINT (929963.379 1932051.62)
4,JDQ4MTYyMiM1MSMkMiMkMCMkMDAkMzgxMzUxIzQxIyQxIy...,(사)대한결핵협회경기도지부부설복십자의원,31,의원,310000,경기,310603,수원팔달구,매산로3가,16456,...,0,0,0,0,0,0,0,127.008966,37.271984,POINT (956465.71 1919343.077)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
78769,JDQ4MTg4MSM1MSMkMiMkNCMkMDAkNDgxMTkxIzMxIyQxIy...,힘튼의원,31,의원,110000,서울,110015,은평구,응암동,3462,...,0,0,0,0,0,0,0,126.926666,37.601101,POINT (949391.135 1955897.332)
78770,JDQ4MTYyMiM2MSMkMSMkMiMkNzIkMzgxMTkxIzExIyQxIy...,힘플란트치과의원,51,치과의원,340000,충남,340500,아산시,둔포면,31406,...,0,0,0,0,0,0,0,127.057823,36.923728,POINT (960617.439 1880687.566)
78771,JDQ4MTYyMiM2MSMkMSMkMiMkNzIkMzgxOTYxIzExIyQxIy...,힘플란트치과의원,51,치과의원,250000,대전,250003,대전서구,내동,35314,...,0,0,0,0,0,0,0,127.383241,36.332062,POINT (989521.106 1814971.289)
78772,JDQ4MTYyMiM3MSMkMSMkMCMkOTkkMzgxMzUxIzIxIyQxIy...,힘플러스병원,21,병원,360000,전남,360400,순천시,조례동,57946,...,0,0,0,0,0,0,0,127.519845,34.960448,POINT (1001811.778 1662842.094)


In [26]:
cols_keep = ["EMD_CD", "EMD_KOR_NM"]
df_inst_with_emd = gpd.sjoin(
    gdf_inst,
    gdf_emd[cols_keep + ["geometry"]],
    how="left",
    predicate="within"
).drop(columns=["index_right"])
inst_loc_tmp = df_inst_with_emd[['암호화요양기호', '요양기관명','시도코드명','시군구코드명','EMD_CD','종별코드명', '좌표(X)', '좌표(Y)']].copy()
inst_loc_tmp['종별코드명'] = inst_loc_tmp['종별코드명'].astype(str).str.strip()
loc_hosp  = inst_loc_tmp[inst_loc_tmp['종별코드명'].isin(grp_hosp)]
loc_bogun = inst_loc_tmp[inst_loc_tmp['종별코드명'].isin(grp_bogun)]
loc_bogun

,암호화요양기호,요양기관명,시도코드명,시군구코드명,EMD_CD,종별코드명,좌표(X),좌표(Y)
867,JDQ4MTYyMiM3MSMkMSMkMCMkOTIkMzgxOTYxIzIxIyQyIy...,가거도보건지소,전남,신안군,46910360,보건지소,125.126634,34.052378
873,JDQ4MTYyMiM1MSMkMSMkNCMkOTIkMzgxOTYxIzExIyQxIy...,가곡보건지소,강원,삼척시,51230350,보건지소,129.205182,37.149763
909,JDQ4MTYyMiM1MSMkMSMkOCMkOTIkMzgxOTYxIzExIyQxIy...,가덕면 보건지소,충북,청주상당구,43111330,보건지소,127.548391,36.553416
974,JDQ4MTYyMiM4MSMkMSMkMCMkOTIkMzgxOTYxIzExIyQxIy...,가산면보건지소,경북,칠곡군,47850330,보건지소,128.539051,36.084404
975,JDQ4MTYyMiM1MSMkMSMkMCMkOTIkMzgxOTYxIzExIyQyIy...,가산보건지소,경기,포천시,41650330,보건지소,127.186376,37.848708
...,...,...,...,...,...,...,...,...
78543,JDQ4MTYyMiM4MSMkMSMkNCMkOTIkMzgxOTYxIzExIyQxIy...,휴천면보건지소,경남,함양군,48870320,보건지소,127.750459,35.479959
78581,JDQ4MTYyMiM3MSMkMSMkMCMkOTIkMzgxOTYxIzExIyQxIy...,흑산면보건지소,전남,신안군,46910360,보건지소,125.442788,34.683710
78603,JDQ4MTYyMiM1MSMkMSMkNCMkOTIkMzgxOTYxIzExIyQxIy...,흥업면보건지소,강원,원주시,51130370,보건지소,127.921177,37.302613
78612,JDQ4MTYyMiM1MSMkMSMkMCMkOTIkMzgxOTYxIzExIyQxIy...,흥천보건지소,경기,여주시,41670320,보건지소,127.543124,37.331333


In [27]:
# 현재 공보의 배치안 추가
loc_bogun = pd.merge(loc_bogun, df_current_allocation[['요양기관명', '현재공보의배치여부', '시군구코드명']], how = 'left', on = ['요양기관명', '시군구코드명'])
loc_bogun['현재공보의배치여부'] = loc_bogun['현재공보의배치여부'].fillna(0)

In [28]:
# 공보의 배치 기준에 따라 배치 가능한 곳 필터링
# 광역시 소속이 아닌 곳
cond_not_metro = ~loc_bogun["시도코드명"].isin(
    ["서울", "부산", "대구", "인천", "광주", "대전", "울산", "세종"]
)

# 광역시 소속이며, 군 소속인곳
cond_metro_and_goon = (
    loc_bogun["시도코드명"].isin(
        ["서울", "부산", "대구", "인천", "광주", "대전", "울산", "세종"]
    )
    & loc_bogun["시군구코드명"].str.endswith("군")
)

# 시 보건소 및 시 보건지소 중 제외 지역
si_exclude = ~loc_bogun["시군구코드명"].str.contains("|".join(
    ["수원", "성남", "의정부", "안양", "부천", "광명", "고양", "과천", "구리", "오산", "시흥", "군포", "의왕", "하남"])
)

# 비연륙도서 제외
nobridge_island_exclude = ~loc_bogun["암호화요양기호"].isin(df_bogun_island['암호화요양기호'])

# 필터링 적용
base_mask = (cond_not_metro | cond_metro_and_goon) & si_exclude & nobridge_island_exclude
final_mask = base_mask
# keep_current = loc_bogun["현재공보의배치여부"] == 1 # 근데 현재 배치되어있으면 포함
# final_mask = base_mask | keep_current
loc_bogun_filtered = loc_bogun[final_mask].copy()
loc_bogun_filtered # 최종 배치하는 보건소/보건지소 목록

,암호화요양기호,요양기관명,시도코드명,시군구코드명,EMD_CD,종별코드명,좌표(X),좌표(Y),현재공보의배치여부
1,JDQ4MTYyMiM1MSMkMSMkNCMkOTIkMzgxOTYxIzExIyQxIy...,가곡보건지소,강원,삼척시,51230350,보건지소,129.205182,37.149763,1.0
2,JDQ4MTYyMiM1MSMkMSMkOCMkOTIkMzgxOTYxIzExIyQxIy...,가덕면 보건지소,충북,청주상당구,43111330,보건지소,127.548391,36.553416,1.0
3,JDQ4MTYyMiM4MSMkMSMkMCMkOTIkMzgxOTYxIzExIyQxIy...,가산면보건지소,경북,칠곡군,47850330,보건지소,128.539051,36.084404,0.0
4,JDQ4MTYyMiM1MSMkMSMkMCMkOTIkMzgxOTYxIzExIyQyIy...,가산보건지소,경기,포천시,41650330,보건지소,127.186376,37.848708,0.0
5,JDQ4MTYyMiM2MSMkMSMkMiMkOTIkMzgxOTYxIzExIyQxIy...,가야곡보건지소,충남,논산시,44230400,보건지소,127.158798,36.136807,1.0
...,...,...,...,...,...,...,...,...,...
1550,JDQ4MTYyMiM4MSMkMSMkMCMkOTIkMzgxOTYxIzExIyQxIy...,효자면.은풍면통합보건지소,경북,예천군,47900430,보건지소,128.443782,36.745597,0.0
1551,JDQ4MTYyMiM4MSMkMSMkNCMkOTIkMzgxOTYxIzExIyQxIy...,휴천면보건지소,경남,함양군,48870320,보건지소,127.750459,35.479959,1.0
1553,JDQ4MTYyMiM1MSMkMSMkNCMkOTIkMzgxOTYxIzExIyQxIy...,흥업면보건지소,강원,원주시,51130370,보건지소,127.921177,37.302613,0.0
1554,JDQ4MTYyMiM1MSMkMSMkMCMkOTIkMzgxOTYxIzExIyQxIy...,흥천보건지소,경기,여주시,41670320,보건지소,127.543124,37.331333,0.0


In [29]:
out_path = Path("intermediate_result/loc_bogun_filtered.csv")
loc_bogun_filtered.to_csv(
    out_path,
    index=False,
    encoding="utf-8-sig"
)

In [30]:
# gdf_bogun, gdf_hosp 생성
gdf_bogun = gpd.GeoDataFrame(
    loc_bogun_filtered,
    geometry=gpd.points_from_xy(loc_bogun_filtered["좌표(X)"], loc_bogun_filtered["좌표(Y)"]),
    crs="EPSG:4326"
)
gdf_hosp = gpd.GeoDataFrame(
    loc_hosp,
    geometry=gpd.points_from_xy(loc_hosp["좌표(X)"], loc_hosp["좌표(Y)"]),
    crs="EPSG:4326"
)

G = nx.compose_all([G_한국, G_울릉, G_제주])

In [31]:
RESULT_DIR = ROOT_DIR / 'intermediate_result'

In [32]:
gdf_grid_pop = gdf_grid_pop.set_geometry("grid_centroid")
gdf_grid_pop = gdf_grid_pop.to_crs("EPSG:4326") # 현재 도로망상의 node는 EPSG:4326으로 저장되어 있으므로 변환했다가 nearest node를 구한 후 다시 바꿔줌

# 격자 → 도로망상 노드
grid_nodes = ox.nearest_nodes(
    G,
    gdf_grid_pop.geometry.x.values,
    gdf_grid_pop.geometry.y.values
)

# 보건소 → 노드
inst_nodes = ox.nearest_nodes(
    G,
    gdf_bogun.geometry.x.values,
    gdf_bogun.geometry.y.values
)

In [33]:
candidate_nodes = set(grid_nodes) | set(inst_nodes)
G_sub = G.subgraph(candidate_nodes).copy()
G_sub = ox.project_graph(G_sub, to_crs="EPSG:5179")
gdf_bogun = gpd.GeoDataFrame(
    loc_bogun_filtered,
    geometry=gpd.points_from_xy(loc_bogun_filtered["좌표(X)"], loc_bogun_filtered["좌표(Y)"]),
    crs="EPSG:4326"
).to_crs("EPSG:5179")
gdf_hosp = gpd.GeoDataFrame(
    loc_hosp,
    geometry=gpd.points_from_xy(loc_hosp["좌표(X)"], loc_hosp["좌표(Y)"]),
    crs="EPSG:4326"
).to_crs("EPSG:5179")

In [34]:
nodes_gdf = ox.graph_to_gdfs(G_sub, nodes=True, edges=False)

In [35]:
gdf_grid_pop.geometry = gdf_grid_pop.geometry.to_crs("EPSG:5179")

In [36]:
nodes_inst = nodes_gdf.loc[inst_nodes, ["x", "y"]]
nodes_grid = nodes_gdf.loc[grid_nodes, ["x", "y"]]

In [37]:
gdf_bogun["node_id"] = inst_nodes
xy_inst = np.c_[gdf_bogun.geometry.x.values, gdf_bogun.geometry.y.values] # 각 기관의 실제 좌표
xy_node_inst = nodes_inst.reindex(gdf_bogun["node_id"]).to_numpy() # 각 기관이 매핑된 도로망 노드 좌표 가져오기
inst_offset = np.hypot(xy_inst[:,0] - xy_node_inst[:,0], xy_inst[:,1] - xy_node_inst[:,1]) # 기관 - 노드 간 거리 계산
gdf_bogun["offset_to_node_m"] = inst_offset

In [38]:
xy_grid = np.c_[gdf_grid_pop.geometry.x.values, gdf_grid_pop.geometry.y.values] # 각 격자점 centroid의 실제 좌표
xy_node_grid = nodes_grid.reindex(grid_nodes).to_numpy() # 각 격자점이 매핑된 도로망 노드 좌표 가져오기
grid_offset = np.hypot(xy_grid[:,0] - xy_node_grid[:,0], xy_grid[:,1] - xy_node_grid[:,1]) # 기관 - 노드 간 거리 계산

In [39]:
gdf_grid_pop["node_id"] = grid_nodes
gdf_grid_pop["offset_to_node_m"] = grid_offset

In [40]:
max(inst_offset)

np.float64(408.86463649634163)

In [41]:
max(grid_offset)

np.float64(142858.48507242207)

In [42]:
gdf_grid_pop[gdf_grid_pop['offset_to_node_m'] == max(grid_offset)]

,GRID_500M_,geometry,grid_centroid,population,EMD_CD,법정동코드,coefficient,adjusted_population,node_id,offset_to_node_m
474,가아46a00b,"POLYGON ((746000 2000500, 746000 2001000, 7465...",POINT (746250 2000750),0,28720330,28720330,1.048637,0.0,1011918426,142858.485072


In [43]:
out_path = Path("intermediate_result/gdf_emd.csv")
gdf_emd.to_csv(
    out_path,
    index=False,
    encoding="utf-8-sig"
)

In [44]:
out_path = Path("intermediate_result/gdf_bogun.csv")
gdf_bogun.to_csv(
    out_path,
    index=False,
    encoding="utf-8-sig"
)

In [46]:
gdf_bogun.to_file(RESULT_DIR / "gdf_bogun.gpkg", driver = "GPKG")
gdf_hosp.to_file(RESULT_DIR / "gdf_hosp.gpkg", driver = "GPKG")
gdf_grid_pop.to_file(RESULT_DIR / "gdf_grid_pop.gpkg", driver="GPKG")

ValueError: GeoDataFrame contains multiple geometry columns but GeoDataFrame.to_file supports only a single geometry column. Use a GeoDataFrame.to_parquet or GeoDataFrame.to_feather, drop additional geometry columns or convert them to a supported format like a well-known text (WKT) using `GeoSeries.to_wkt()`.

In [46]:
gdf_grid_pop_copy = gdf_grid_pop.drop(columns='geometry')

In [47]:
out_path = Path("intermediate_result/gdf_grid_pop.csv")
gdf_grid_pop.to_csv(
    out_path,
    index=False,
    encoding="utf-8-sig"
)